# 08 — Sync

Unified sync workflow: preview what would change (read-only), then run
incremental writes. Merges the old `data_refresh` + `sync_preview` notebooks.

- **Part A (Preview):** Auth check, DB state, fetch playlists, sync delta, API
  estimate, ignore-list management — read-only, 1 API call.
- **Part B (Sync):** Limits config, upsert playlists, sync tracks, fill audio
  features, fill artists, verify.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import httpx
import polars as pl

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from etl.sync import plan_sync
from spotify.auth import CACHE_PATH, SpotifyAuth
from spotify.client import SpotifyClient
from spotify.fetch import fetch_my_playlists
from utils.config import settings
from utils.logging import configure_logging

configure_logging()

TABLES = ["playlists", "tracks", "audio_features", "artists", "track_artists", "playlist_tracks"]

---
# Part A — Preview (read-only)

## 1. Spotify Auth Check

In [ ]:
# Token status — cache existence, expiry, live /v1/me probe
if not CACHE_PATH.exists():
    print(f"NO CACHE at {CACHE_PATH} — run `make auth` to authenticate.")
else:
    data = json.loads(CACHE_PATH.read_text())
    at = data.get("access_token", "")
    exp = data.get("expires_at", 0)
    expired = time.time() > exp - 60
    print(f"Cache:   {CACHE_PATH}")
    print(f"Token:   {at[:20]}... ({len(at)} chars)")
    print(f"Expired: {expired}")

    # Live probe
    r = httpx.get("https://api.spotify.com/v1/me", headers={"Authorization": f"Bearer {at}"})
    if r.status_code == 200:
        print(f"User:    {r.json().get('display_name', '?')} (200 OK)")
    else:
        print(f"Probe failed: {r.status_code} — token may need refresh")

In [ ]:
# Force-refresh token (run only if probe above failed)
# token = SpotifyAuth().force_refresh()
# print(f"New token: {token[:20]}...")

## 2. DB State

In [ ]:
conn = get_connection(read_only=True)

# Table row counts
print("Table row counts:")
for table in TABLES:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:20s} {n:>7,}")

n_profile = conn.execute("SELECT COUNT(*) FROM track_profile").fetchone()[0]
print(f"  {'track_profile':20s} {n_profile:>7,}  (view)")

In [ ]:
# Gap audit — missing audio features and artists by playlist
missing_audio_by_playlist = conn.execute("""
    SELECT p.playlist_name, COUNT(*) AS missing_audio
    FROM tracks t
    LEFT JOIN audio_features af USING (track_id)
    JOIN playlist_tracks pt USING (track_id)
    JOIN playlists p USING (playlist_id)
    WHERE af.track_id IS NULL
    GROUP BY p.playlist_name
    ORDER BY missing_audio DESC
""").pl()

missing_audio_total = conn.execute("""
    SELECT COUNT(*) FROM tracks t
    LEFT JOIN audio_features af USING (track_id)
    WHERE af.track_id IS NULL
""").fetchone()[0]

missing_artists_total = conn.execute("""
    SELECT COUNT(DISTINCT ta.artist_id) FROM track_artists ta
    LEFT JOIN artists a USING (artist_id)
    WHERE a.artist_id IS NULL
""").fetchone()[0]

print(f"Tracks missing audio features: {missing_audio_total}")
print(f"Artists missing data:          {missing_artists_total}")

if not missing_audio_by_playlist.is_empty():
    print("\nMissing audio features by playlist:")
    print(missing_audio_by_playlist)

## 3. Fetch Playlists from API

One API call with a 1-hour `/tmp` file cache.

In [ ]:
_PLAYLIST_CACHE = Path("/tmp/spotify_playlists_preview.json")
_CACHE_TTL = 3600  # 1 hour

def _cache_fresh() -> bool:
    return _PLAYLIST_CACHE.exists() and (time.time() - _PLAYLIST_CACHE.stat().st_mtime) < _CACHE_TTL

if _cache_fresh():
    raw_playlists = json.loads(_PLAYLIST_CACHE.read_text())
    print(f"Loaded {len(raw_playlists)} playlists from cache (< 1hr old)")
else:
    client = SpotifyClient()
    raw_playlists = fetch_my_playlists(client)
    _PLAYLIST_CACHE.write_text(json.dumps(raw_playlists))
    print(f"Fetched {len(raw_playlists)} playlists from API (cached to {_PLAYLIST_CACHE})")

# To bust cache: _PLAYLIST_CACHE.unlink()

## 4. Sync Delta

`plan_sync()` classifies each playlist as new / stale / current / excluded.

In [ ]:
items = plan_sync(conn, raw_playlists)

delta = pl.DataFrame([{
    "playlist_name": i.playlist_name,
    "status": i.playlist_status,
    "spotify_tracks": i.spotify_track_count,
    "db_tracks": i.db_track_count,
    "delta": i.spotify_track_count - i.db_track_count,
    "needs_sync": i.needs_sync,
} for i in items])

print(f"{len(items)} playlists total:")
for status in ["active", "archived", "excluded"]:
    n = delta.filter(pl.col("status") == status).height
    print(f"  {status:12s} {n}")

to_sync = delta.filter(pl.col("needs_sync"))
print(f"\nNeed sync: {to_sync.height}")
delta.sort("needs_sync", descending=True)

## 5. API Call Estimate

In [ ]:
# Spotify API page sizes
TRACKS_PER_PAGE = 100
AUDIO_PER_PAGE = 100
ARTISTS_PER_PAGE = 50

sync_items = [i for i in items if i.needs_sync]

playlist_calls = len(sync_items)
track_fetch_calls = sum(
    max(1, -(-i.spotify_track_count // TRACKS_PER_PAGE)) for i in sync_items
)
audio_calls = max(1, -(-missing_audio_total // AUDIO_PER_PAGE)) if missing_audio_total else 0
artist_calls = max(1, -(-missing_artists_total // ARTISTS_PER_PAGE)) if missing_artists_total else 0

total = 1 + track_fetch_calls + audio_calls + artist_calls  # +1 for fetch_my_playlists

print("Estimated API calls:")
print(f"  fetch_my_playlists:    1")
print(f"  fetch_playlist_tracks: {track_fetch_calls} ({len(sync_items)} playlists)")
print(f"  fetch_audio_features:  {audio_calls} ({missing_audio_total} tracks)")
print(f"  fetch_artist_features: {artist_calls} ({missing_artists_total} artists)")
print(f"  ─────────────────────────")
print(f"  TOTAL:                 {total}")

## 6. Ignore List Management

Exclude or re-include playlists from sync. Edit the name, then run the cell.

In [ ]:
# Exclude a playlist from sync
# EXCLUDE_NAME = "Playlist Name"
# conn.close()
# write_conn = get_connection(read_only=False)
# write_conn.execute(
#     "UPDATE playlists SET include_in_refresh=FALSE WHERE playlist_name=?",
#     [EXCLUDE_NAME],
# )
# write_conn.close()
# conn = get_connection(read_only=True)
# print(f"Excluded: {EXCLUDE_NAME}")

In [ ]:
# Re-include a playlist in sync
# INCLUDE_NAME = "Playlist Name"
# conn.close()
# write_conn = get_connection(read_only=False)
# write_conn.execute(
#     "UPDATE playlists SET include_in_refresh=TRUE WHERE playlist_name=?",
#     [INCLUDE_NAME],
# )
# write_conn.close()
# conn = get_connection(read_only=True)
# print(f"Re-included: {INCLUDE_NAME}")

---
# Part B — Sync (write path)

## 7. Limits Config

| Param | Effect |
|---|---|
| `MAX_PLAYLISTS` | Only sync tracks for N playlists |
| `MAX_TRACKS` | Cap total tracks fetched |
| `AUDIO_LIMIT` | Cap audio feature fetches |
| `ARTIST_LIMIT` | Cap artist feature fetches |

Set to `None` for no limit (full sync).

In [ ]:
from etl.sync import sync_artist_features, sync_audio_features, sync_tracks, upsert_playlists

MAX_PLAYLISTS: int | None = None
MAX_TRACKS:    int | None = None
AUDIO_LIMIT:   int | None = None
ARTIST_LIMIT:  int | None = None

conn.close()
write_conn = get_connection(read_only=False)
init_schema(write_conn)
client = SpotifyClient()

print("Write connection open. Limits:")
print(f"  MAX_PLAYLISTS = {MAX_PLAYLISTS}")
print(f"  MAX_TRACKS    = {MAX_TRACKS}")
print(f"  AUDIO_LIMIT   = {AUDIO_LIMIT}")
print(f"  ARTIST_LIMIT  = {ARTIST_LIMIT}")

## 8. Upsert Playlists

In [ ]:
upsert_playlists(write_conn, items)
n_playlists = write_conn.execute("SELECT COUNT(*) FROM playlists").fetchone()[0]
print(f"Playlists in DB: {n_playlists}")

## 9. Sync Tracks

In [ ]:
all_tracks = sync_tracks(
    write_conn, client, items,
    max_playlists=MAX_PLAYLISTS, max_tracks=MAX_TRACKS,
)
print(f"Tracks synced: {len(all_tracks)}")

## 10. Fill Missing Audio Features

In [ ]:
n_audio = sync_audio_features(write_conn, client, limit=AUDIO_LIMIT)
print(f"Audio features filled: {n_audio}")

## 11. Fill Missing Artist Data

In [ ]:
n_artists = sync_artist_features(write_conn, client, limit=ARTIST_LIMIT)
print(f"Artist features filled: {n_artists}")

## 12. Verify

In [ ]:
write_conn.close()
conn = get_connection(read_only=True)

print("Final table counts:")
for table in TABLES:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:20s} {n:>7,}")

n_profile = conn.execute("SELECT COUNT(*) FROM track_profile").fetchone()[0]
print(f"  {'track_profile':20s} {n_profile:>7,}  (view)")

conn.close()
print("\nAll connections closed.")